In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('chats_sample_baseline.csv')

In [ ]:
df.head()

In [ ]:
# Convert the 'timestamp' column to datetime objects
df['timestamp'] = pd.to_datetime(df['timestamp'], format='%d-%m-%Y %H:%M', errors='coerce')

# Extract date and time into separate columns
df['date'] = df['timestamp'].dt.date
df['time'] = df['timestamp'].dt.time



In [ ]:
# Display the first few rows and column information to confirm the conversion
display(df.head())
display(df.info())
df['label'].value_counts()

In [ ]:
# Extract the hour from the 'time' column
df['hour'] = df['time'].apply(lambda x: x.hour)

# Display the first few rows with the new 'hour' column
display(df.head())

Now that we have a numerical representation of the time (hour), we can visualize its distribution across different labels. A violin plot is a good choice here as it shows the distribution shape, median, and interquartile range for each category.

In [ ]:
df_filtered = df[df['label'] != -1]
display(df_filtered['label'].value_counts())

In [ ]:
plt.figure(figsize=(10, 6))
sns.violinplot(x='label', y='hour', data=df_filtered, inner='box', hue='label', palette='Set2', legend=False)
plt.title('Distribution of Message Hours by Label')
plt.xlabel('Label')
plt.ylabel('Hour of Day')
plt.yticks(np.arange(0, 24, 2)) # Show hours in steps of 2
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

Violin plot observations:
1. Urgent messages (label = 1) show bimodal peaks around early morning (around 3 AM) and early afternoon ( around 1 PM), suggesting urgency is not confined to typical active hours.
2. Late-night/early-morning messages (0–5 AM) appear disproportionately associated with urgency, indicating time-of-day can be a strong signal for DND override.
3. Non-urgent messages (label = 0) cluster more around evening/night hours (~11 PM), suggesting higher noise during these periods.
4. messages coming at early morning/late night times (between 00:00 and 04:00) are more likely emergencies


In [ ]:
sender_label_counts = df_filtered.groupby(['sender', 'label']).size().unstack(fill_value=0)
sender_label_proportions = sender_label_counts.apply(lambda x: x / x.sum(), axis=1)

# Sort by the proportion of label 1 messages for better visualization
sender_label_proportions = sender_label_proportions.sort_values(by=1, ascending=False)

plt.figure(figsize=(12, 7))
sender_label_proportions.plot(kind='bar', stacked=True, colormap='viridis', figsize=(12, 7))
plt.title('Proportion of Message Labels by Sender')
plt.xlabel('Sender')
plt.ylabel('Proportion of Messages')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Label')
plt.tight_layout()
plt.show()


1. The chart shows that some people message exclusively in cases of emergencies and these people should be considered more likely to bypass (e.g. Dr. <>)
2. Alternatively there is no emergency labels associated with some peoeple. These people are less likely to approach us when in need of help.
3. The distribution of senders can be regarded as 'low urgency senders', 'high urgency senders' and 'mixed urgency senders'. These should be weighted to prioritize bypass.

In [ ]:
sender_hour_counts = df_filtered.groupby(['sender', 'hour']).size().unstack(fill_value=0)

plt.figure(figsize=(15, 8))
sns.heatmap(sender_hour_counts, cmap='viridis', linewidths=.5)
plt.title('Message Frequency by Sender and Hour of Day')
plt.xlabel('Hour of Day')
plt.ylabel('Sender')
plt.show()

1. The heatmap shows frequency of messages categorized by hour of day. There is disproportionately more messages from person 1, 14, and 3. However, they fall under the "low urgency senders" and "mixed urgency senders" labels with an inclination towards zero urgent messages.
2. The "high urgency senders" category seems to send less messages (e.g. persons 10, 7, 8, 5, and 9 who send no more than 5 messages at any given hour).
3. The chart showcases a negative correlation of senders' message frequency and urgency.

In [ ]:
urgent_messages = df_filtered[df_filtered['label'] == 1]
sender_hour_urgency_counts = urgent_messages.groupby(['sender', 'hour']).size().unstack(fill_value=0)

plt.figure(figsize=(15, 8))
sns.heatmap(sender_hour_urgency_counts, cmap='Reds', linewidths=.5)
plt.title('Urgent Message Frequency by Sender and Hour of Day (Label = 1)')
plt.xlabel('Hour of Day')
plt.ylabel('Sender')
plt.show()

In [ ]:
plt.figure(figsize=(15, 8))
sender_label_counts.plot(kind='bar', figsize=(15, 8), color=['skyblue', 'salmon'])
plt.title('Message Counts by Sender and Urgency Label')
plt.xlabel('Sender')
plt.ylabel('Number of Messages')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Label')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

The two plots above give us the following info:
1. "High urgency senders" have very little message frequency implying that they only message in case of emergencies.
2. "Mixed urgency senders" generally have a disproportionately higher frequency of non-urgent messages

In [ ]:
df_for_plot = df_filtered.copy() # Explicitly create a copy to avoid SettingWithCopyWarning
df_for_plot['message_length'] = df_for_plot['message'].astype(str).apply(len)

plt.figure(figsize=(10, 6))
sns.violinplot(x='label', y='message_length', data=df_for_plot, inner='box', hue='label', palette='viridis', legend=False)
plt.title('Distribution of Message Length by Urgency Label')
plt.xlabel('Urgency Label (0: Non-Urgent, 1: Urgent)')
plt.ylabel('Message Length')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

1. Non-Urgent (Label 0): Often shows a wider distribution, with a peak at shorter lengths (e.g., quick replies, casual chats) but also a long tail extending to very long messages (e.g., detailed explanations, extended conversations).
2. Urgent (Label 1): Might show a more concentrated distribution. Sometimes, urgent messages are very short and to the point ('Help!', 'Urgent!'). Other times, they might be moderately short but descriptive, avoiding excessive detail to convey information quickly.